# Practical Session 05
## Part I - Instructor-led mini-lab
### Example 1 - Timing a numerical kernel as an experiment

In this example, we use a small nonlinear finite-difference residual as a representative scientific workload.
The goal is not to make the code fast immediately. The goal is to establish a clear baseline, validate a candidate implementation, and then measure it under controlled conditions.

In [ ]:
import io
import timeit
import cProfile
import pstats
from pstats import SortKey
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from numba import njit
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False

print("Numba available:", HAS_NUMBA)

In [ ]:
def make_problem(n):
    """Create a one-dimensional field and grid spacing."""
    x = np.linspace(-10.0, 10.0, n)
    dx = x[1] - x[0]
    u = np.tanh(x) + 0.05 * np.sin(3.0 * x)
    return x, u, dx


def residual_reference(u, dx):
    """Clear Python-loop baseline for a nonlinear residual."""
    r = np.zeros_like(u)
    inv_dx2 = 1.0 / dx**2

    for i in range(1, u.size - 1):
        u_xx = (u[i - 1] - 2.0*u[i] + u[i + 1]) * inv_dx2
        r[i] = -u_xx + u[i]*(u[i]**2 - 1.0)

    return r


def residual_vectorized(u, dx):
    """Vectorized candidate implementation of the same residual."""
    r = np.zeros_like(u)
    inv_dx2 = 1.0 / dx**2

    r[1:-1] = (
        -(u[:-2] - 2.0*u[1:-1] + u[2:]) * inv_dx2
        + u[1:-1]*(u[1:-1]**2 - 1.0)
    )

    return r


def benchmark_call(call, *, number=1, repeat=5):
    """Return repeated timing samples in seconds per call."""
    samples = timeit.repeat(call, number=number, repeat=repeat)
    return np.array(samples) / number


def timing_summary(name, samples):
    return {
        "implementation": name,
        "best_s": samples.min(),
        "median_s": np.median(samples),
        "std_s": samples.std(ddof=1) if samples.size > 1 else 0.0,
    }

In [ ]:
x, u, dx = make_problem(50_000)

r_ref = residual_reference(u, dx)
r_vec = residual_vectorized(u, dx)

max_difference = np.max(np.abs(r_ref - r_vec))
print("max |reference - vectorized|:", max_difference)
assert np.allclose(r_ref, r_vec)

samples_ref = benchmark_call(lambda: residual_reference(u, dx), number=2, repeat=5)
samples_vec = benchmark_call(lambda: residual_vectorized(u, dx), number=50, repeat=7)

timings = pd.DataFrame([
    timing_summary("reference loop", samples_ref),
    timing_summary("vectorized", samples_vec),
])

timings["speedup_vs_reference"] = timings.loc[0, "median_s"] / timings["median_s"]
timings

Questions:

1. Which quantity was measured: grid construction, residual evaluation, or the whole workflow?
2. Why did we validate `residual_vectorized` before comparing its runtime?
3. Which timing number would you report: best, median, or the full spread?

### Example 2 - Scaling across problem sizes

A timing at one size is only a local statement. A scaling experiment asks how the cost changes when the numerical resolution changes.
Here the input size is also a numerical parameter: increasing `n` decreases the grid spacing `dx`.

In [ ]:
sizes = np.array([1_000, 3_000, 10_000, 30_000, 100_000])
rows = []

for n in sizes:
    x, u, dx = make_problem(n)

    number_vec = 200 if n <= 10_000 else 30
    t_vec = benchmark_call(lambda u=u, dx=dx: residual_vectorized(u, dx),
                           number=number_vec, repeat=5)
    rows.append({
        "n": n,
        "implementation": "vectorized",
        "best_s": t_vec.min(),
        "median_s": np.median(t_vec),
        "std_s": t_vec.std(ddof=1),
    })

    # The reference loop is intentionally measured only up to moderate sizes.
    if n <= 30_000:
        number_ref = 5 if n <= 10_000 else 1
        t_ref = benchmark_call(lambda u=u, dx=dx: residual_reference(u, dx),
                               number=number_ref, repeat=3)
        rows.append({
            "n": n,
            "implementation": "reference loop",
            "best_s": t_ref.min(),
            "median_s": np.median(t_ref),
            "std_s": t_ref.std(ddof=1),
        })

scaling = pd.DataFrame(rows)
scaling[["n", "implementation", "best_s", "median_s", "std_s"]]

In [ ]:
fig, ax = plt.subplots(figsize=(5.8, 3.4))

for name, group in scaling.groupby("implementation"):
    group = group.sort_values("n")
    ax.loglog(group["n"], group["median_s"], "o-", label=name)

ax.set_xlabel(r"number of grid points $n$")
ax.set_ylabel("median time per residual call [s]")
ax.set_title("Scaling of two residual implementations")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

vec = scaling.query("implementation == 'vectorized'").sort_values("n")
p_est = np.polyfit(np.log(vec["n"]), np.log(vec["median_s"]), deg=1)[0]
print(f"estimated scaling exponent for vectorized implementation: {p_est:.2f}")

Questions:

1. Does the vectorized implementation mainly change the scaling law or the constant factor?
2. Why might the smallest size be a poor representation of a real simulation?
3. What would you measure next if the largest run was still too slow?

### Example 3 - Profiling a workflow with call structure

Timing tells us how long the computation takes. Profiling tells us where the time is spent.
Here we intentionally use a slow residual with many small Python-level calls, then use the profile to justify one change.

In [ ]:
def local_force_scalar(value):
    return value * (value**2 - 1.0)


def residual_many_small_calls(u, dx):
    r = np.zeros_like(u)
    inv_dx2 = 1.0 / dx**2

    for i in range(1, u.size - 1):
        u_xx = (u[i - 1] - 2.0*u[i] + u[i + 1]) * inv_dx2
        r[i] = -u_xx + local_force_scalar(u[i])

    return r


def step_slow(u, dx, dt):
    return u - dt * residual_many_small_calls(u, dx)


def run_simulation_slow(n=4_000, n_steps=80, dt=1.0e-6):
    x, u, dx = make_problem(n)

    for _ in range(n_steps):
        u = step_slow(u, dx, dt)

    return u


profiler = cProfile.Profile()
profiler.enable()
u_slow = run_simulation_slow(n=4_000, n_steps=80)
profiler.disable()

stream = io.StringIO()
stats = pstats.Stats(profiler, stream=stream)
stats.strip_dirs()
stats.sort_stats(SortKey.CUMULATIVE)
stats.print_stats(10)

print(stream.getvalue())

In [ ]:
def step_fast(u, dx, dt):
    return u - dt * residual_vectorized(u, dx)


def run_simulation_fast(n=4_000, n_steps=80, dt=1.0e-6):
    x, u, dx = make_problem(n)

    for _ in range(n_steps):
        u = step_fast(u, dx, dt)

    return u


u_fast = run_simulation_fast(n=4_000, n_steps=80)
print("max difference after simulation:", np.max(np.abs(u_slow - u_fast)))
assert np.allclose(u_slow, u_fast)

t_slow = benchmark_call(lambda: run_simulation_slow(n=4_000, n_steps=80), number=1, repeat=3)
t_fast = benchmark_call(lambda: run_simulation_fast(n=4_000, n_steps=80), number=1, repeat=5)

profile_decision = pd.DataFrame([
    timing_summary("slow call structure", t_slow),
    timing_summary("vectorized residual", t_fast),
])
profile_decision["speedup_vs_slow"] = profile_decision.loc[0, "median_s"] / profile_decision["median_s"]
profile_decision

Questions:

1. Which functions dominate cumulative time in the profile?
2. Which functions dominate internal time?
3. Was the change justified by the profile, or was it only a guess?

### Example 4 - Memory-aware updates and optional JIT compilation

Vectorization is often the simplest useful acceleration strategy, but it can allocate temporary arrays.
For iterative algorithms, a reusable output array can reduce allocation pressure. If the hot path is a simple numerical loop, a compiled kernel may also be a good target.

In [ ]:
def residual_vectorized_out(u, dx, out):
    """Write the residual into an existing output array."""
    inv_dx2 = 1.0 / dx**2

    out[0] = 0.0
    out[-1] = 0.0
    out[1:-1] = (
        -(u[:-2] - 2.0*u[1:-1] + u[2:]) * inv_dx2
        + u[1:-1]*(u[1:-1]**2 - 1.0)
    )

    return out


def evolve_allocating(u0, dx, dt, n_steps):
    u = u0.copy()

    for _ in range(n_steps):
        r = residual_vectorized(u, dx)
        u = u - dt * r

    return u


def evolve_reusing(u0, dx, dt, n_steps):
    u = u0.copy()
    r = np.empty_like(u)

    for _ in range(n_steps):
        residual_vectorized_out(u, dx, r)
        u -= dt * r

    return u


x, u0, dx = make_problem(80_000)
dt = 1.0e-8
n_steps = 80

u_alloc = evolve_allocating(u0, dx, dt, n_steps)
u_reuse = evolve_reusing(u0, dx, dt, n_steps)

print("max difference:", np.max(np.abs(u_alloc - u_reuse)))
assert np.allclose(u_alloc, u_reuse)

print(f"one field array: {u0.nbytes / 1e6:.2f} MB")
print("allocating version creates a new residual array at every step")
print("reusing version keeps one residual work array for the whole loop")

t_alloc = benchmark_call(lambda: evolve_allocating(u0, dx, dt, n_steps), number=1, repeat=3)
t_reuse = benchmark_call(lambda: evolve_reusing(u0, dx, dt, n_steps), number=1, repeat=5)

memory_decision = pd.DataFrame([
    timing_summary("allocating", t_alloc),
    timing_summary("reusing output", t_reuse),
])
memory_decision["speedup_vs_allocating"] = memory_decision.loc[0, "median_s"] / memory_decision["median_s"]
memory_decision

In [ ]:
if HAS_NUMBA:
    @njit
    def residual_numba_out(u, dx, out):
        inv_dx2 = 1.0 / dx**2
        out[0] = 0.0
        out[-1] = 0.0

        for i in range(1, u.size - 1):
            u_xx = (u[i - 1] - 2.0*u[i] + u[i + 1]) * inv_dx2
            out[i] = -u_xx + u[i]*(u[i]**2 - 1.0)

    @njit
    def evolve_numba(u0, dx, dt, n_steps):
        u = u0.copy()
        r = np.empty_like(u)

        for step in range(n_steps):
            residual_numba_out(u, dx, r)
            for i in range(u.size):
                u[i] -= dt * r[i]

        return u

    # First call includes compilation.
    _ = evolve_numba(u0, dx, dt, 1)

    u_jit = evolve_numba(u0, dx, dt, n_steps)
    print("max difference to vectorized/reusing:", np.max(np.abs(u_jit - u_reuse)))
    assert np.allclose(u_jit, u_reuse)

    t_jit = benchmark_call(lambda: evolve_numba(u0, dx, dt, n_steps), number=1, repeat=5)

    jit_decision = pd.DataFrame([
        timing_summary("vectorized/reusing", t_reuse),
        timing_summary("Numba loop", t_jit),
    ])
    jit_decision["speedup_vs_vectorized"] = jit_decision.loc[0, "median_s"] / jit_decision["median_s"]
    display(jit_decision)
else:
    print("Numba is not available in this environment. The JIT part is skipped.")

Questions:

1. What extra complexity did the reusable-output version introduce?
2. Why should the first call to a JIT-compiled function not be mixed with steady-state timing?
3. When would you stop optimizing this example?

## Part II - Independent work

### Task 1 - Benchmark a residual before replacing it

Use the following setup:

In [ ]:
import timeit
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def make_task_problem(n):
    x = np.linspace(-8.0, 8.0, n)
    dx = x[1] - x[0]
    u = np.tanh(x) + 0.02*np.sin(5.0*x)
    return x, u, dx


def residual_baseline(u, dx):
    r = np.zeros_like(u)
    inv_dx2 = 1.0 / dx**2

    for i in range(1, u.size - 1):
        u_xx = (u[i - 1] - 2.0*u[i] + u[i + 1]) * inv_dx2
        r[i] = -u_xx + u[i]*(u[i]**2 - 1.0)

    return r


def benchmark_call(call, *, number=1, repeat=5):
    samples = timeit.repeat(call, number=number, repeat=repeat)
    return np.array(samples) / number


sizes = np.array([1_000, 3_000, 10_000, 30_000])

Your tasks are:

1. Implement `residual_candidate(u, dx)` using vectorized NumPy operations.
2. For each value in `sizes`, check that `residual_candidate` agrees with `residual_baseline`.
3. Benchmark both implementations using repeated timings. Record at least the best and median time per call.
4. Put the results in a small `DataFrame` and make a log-log plot of runtime versus `n`.
5. Write a short interpretation: is the candidate faster, is it still correct, and for which problem sizes is the comparison meaningful?

In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Point 1
# Vectorized candidate implementation
# ------------------------------------------------------------

def residual_candidate(u, dx):
    assert u.ndim == 1
    assert u.size >= 3
    assert dx > 0.0
    assert np.isfinite(u).all()

    r = np.zeros_like(u)
    inv_dx2 = 1.0 / dx**2

    u_xx = (
        u[:-2]
        - 2.0 * u[1:-1]
        + u[2:]
    ) * inv_dx2

    r[1:-1] = (
        -u_xx
        + u[1:-1] * (u[1:-1]**2 - 1.0)
    )

    assert r.shape == u.shape
    assert np.isfinite(r).all()

    return r

# ------------------------------------------------------------
# Point 2 and 3
# Correctness checks and repeated benchmarks
# ------------------------------------------------------------

records = []

for n in sizes:
    x, u, dx = make_task_problem(n)

    # Compute both residuals once before benchmarking.
    r_baseline = residual_baseline(u, dx)
    r_candidate = residual_candidate(u, dx)

    # Correctness check.
    max_abs_difference = np.max(np.abs(r_baseline - r_candidate))

    assert r_baseline.shape == u.shape
    assert r_candidate.shape == u.shape
    assert np.isfinite(r_baseline).all()
    assert np.isfinite(r_candidate).all()
    assert np.allclose(r_candidate, r_baseline)

    # Warm-up calls.
    residual_baseline(u, dx)
    residual_candidate(u, dx)

    # Use more repetitions for smaller problems, fewer for larger ones.
    # The measured quantity is still time per call.
    number = max(3, int(100_000 // n))
    repeat = 7

    baseline_samples = benchmark_call(
        lambda: residual_baseline(u, dx),
        number=number,
        repeat=repeat,
    )

    candidate_samples = benchmark_call(
        lambda: residual_candidate(u, dx),
        number=number,
        repeat=repeat,
    )

    records.append({
        "n": n,
        "method": "baseline_loop",
        "number": number,
        "repeat": repeat,
        "best_s": baseline_samples.min(),
        "median_s": np.median(baseline_samples),
        "std_s": baseline_samples.std(ddof=1),
        "max_abs_difference": 0.0,
    })

    records.append({
        "n": n,
        "method": "candidate_vectorized",
        "number": number,
        "repeat": repeat,
        "best_s": candidate_samples.min(),
        "median_s": np.median(candidate_samples),
        "std_s": candidate_samples.std(ddof=1),
        "max_abs_difference": max_abs_difference,
    })

benchmark_df = pd.DataFrame(records)

display(benchmark_df)

In [ ]:
# ------------------------------------------------------------
# Add speedup information in a compact wide table
# ------------------------------------------------------------

wide = benchmark_df.pivot(
    index="n",
    columns="method",
    values=["best_s", "median_s"],
)

wide["speedup_best"] = (
    wide[("best_s", "baseline_loop")]
    / wide[("best_s", "candidate_vectorized")]
)

wide["speedup_median"] = (
    wide[("median_s", "baseline_loop")]
    / wide[("median_s", "candidate_vectorized")]
)

display(wide)

# ------------------------------------------------------------
# Point 4
# Log-log runtime plot
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.2, 4.0))

for method, table in benchmark_df.groupby("method"):
    table = table.sort_values("n")

    ax.loglog(
        table["n"],
        table["median_s"],
        "o-",
        label=f"{method}, median",
    )

ax.set_xlabel("problem size n")
ax.set_ylabel("runtime per call [s]")
ax.set_title("Residual runtime scaling")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Optional: plot speedup versus n
# ------------------------------------------------------------

speedup_table = wide.reset_index()

fig, ax = plt.subplots(figsize=(6.2, 3.8))

ax.semilogx(
    speedup_table["n"],
    speedup_table["speedup_median"],
    "o-",
)

ax.set_xlabel("problem size n")
ax.set_ylabel("median speedup")
ax.set_title("Vectorized candidate speedup over baseline")
ax.grid(True, which="both", alpha=0.3)

fig.tight_layout()
plt.show()

### Task 2 - Use a profile to justify one change

Use the following setup:

In [ ]:
import io
import cProfile
import pstats
from pstats import SortKey
import timeit

import numpy as np
import pandas as pd


def make_profile_problem(n):
    x = np.linspace(-10.0, 10.0, n)
    dx = x[1] - x[0]
    u = np.tanh(x)
    return x, u, dx


def local_force(value):
    return value * (value**2 - 1.0)


def residual_slow(u, dx):
    r = np.zeros_like(u)
    inv_dx2 = 1.0 / dx**2

    for i in range(1, u.size - 1):
        u_xx = (u[i - 1] - 2.0*u[i] + u[i + 1]) * inv_dx2
        r[i] = -u_xx + local_force(u[i])

    return r


def run_workflow_slow(n=4_000, n_steps=80, dt=1.0e-6):
    x, u, dx = make_profile_problem(n)

    for _ in range(n_steps):
        u = u - dt * residual_slow(u, dx)

    return u

Your tasks are:

1. Profile `run_workflow_slow(n=4_000, n_steps=80)` with `cProfile` and print the 8 most important entries sorted by cumulative time.
2. Identify the main bottleneck and write one hypothesis about why it is expensive.
3. Implement a faster version of the residual or of the workflow.
4. Validate the faster workflow against `run_workflow_slow` for the same input.
5. Benchmark the slow and fast workflows without profiling, report the speedup, and write one sentence explaining whether the change was justified.

In [ ]:
#####################
# Your solution here
#####################


# ------------------------------------------------------------
# Point 1
# Profile the slow workflow
# ------------------------------------------------------------

profiler = cProfile.Profile()

profiler.enable()

u_profile = run_workflow_slow(
    n=4_000,
    n_steps=80,
    dt=1.0e-6,
)

profiler.disable()

stream = io.StringIO()

stats = pstats.Stats(
    profiler,
    stream=stream,
)

stats.strip_dirs()
stats.sort_stats(SortKey.CUMULATIVE)
stats.print_stats(12)

print(stream.getvalue())

assert u_profile.shape == (4_000,)
assert np.isfinite(u_profile).all()

# ------------------------------------------------------------
# Point 2
# Bottleneck hypothesis
# ------------------------------------------------------------

# The profile should show that most time is spent in residual_slow
# and in many calls to local_force.
#
# Hypothesis:
# The workflow is expensive because the residual is computed inside
# a Python loop for every interior grid point and for every time step.
# In addition, local_force is called once per interior point, creating
# a large number of small Python function calls.

In [ ]:
# ------------------------------------------------------------
# Point 3
# Faster residual using vectorized NumPy operations
# ------------------------------------------------------------

def residual_fast(u, dx):
    assert u.ndim == 1
    assert u.size >= 3
    assert dx > 0.0
    assert np.isfinite(u).all()

    r = np.zeros_like(u)
    inv_dx2 = 1.0 / dx**2

    interior = u[1:-1]

    u_xx = (
        u[:-2]
        - 2.0 * interior
        + u[2:]
    ) * inv_dx2

    r[1:-1] = (
        -u_xx
        + interior * (interior**2 - 1.0)
    )

    assert r.shape == u.shape
    assert np.isfinite(r).all()

    return r


def run_workflow_fast(n=4_000, n_steps=80, dt=1.0e-6):
    x, u, dx = make_profile_problem(n)

    for _ in range(n_steps):
        u = u - dt * residual_fast(u, dx)

    return u

# ------------------------------------------------------------
# Point 4
# Validate the faster workflow against the slow workflow
# ------------------------------------------------------------

n_check = 4_000
n_steps_check = 80
dt_check = 1.0e-6

u_slow = run_workflow_slow(
    n=n_check,
    n_steps=n_steps_check,
    dt=dt_check,
)

u_fast = run_workflow_fast(
    n=n_check,
    n_steps=n_steps_check,
    dt=dt_check,
)

assert u_slow.shape == u_fast.shape
assert np.isfinite(u_slow).all()
assert np.isfinite(u_fast).all()

max_abs_difference = np.max(np.abs(u_slow - u_fast))

print("maximum absolute difference:", max_abs_difference)

assert np.allclose(
    u_fast,
    u_slow,
    rtol=1.0e-10,
    atol=1.0e-12,
)

In [ ]:
# ------------------------------------------------------------
# Point 5
# Benchmark slow and fast workflows without profiling
# ------------------------------------------------------------

def benchmark_workflow(call, *, number=1, repeat=5):
    samples = timeit.repeat(
        call,
        number=number,
        repeat=repeat,
    )

    return np.array(samples) / number


# Warm-up calls.
run_workflow_slow(
    n=n_check,
    n_steps=n_steps_check,
    dt=dt_check,
)

run_workflow_fast(
    n=n_check,
    n_steps=n_steps_check,
    dt=dt_check,
)


slow_samples = benchmark_workflow(
    lambda: run_workflow_slow(
        n=n_check,
        n_steps=n_steps_check,
        dt=dt_check,
    ),
    number=1,
    repeat=5,
)

fast_samples = benchmark_workflow(
    lambda: run_workflow_fast(
        n=n_check,
        n_steps=n_steps_check,
        dt=dt_check,
    ),
    number=1,
    repeat=5,
)

benchmark_df = pd.DataFrame([
    {
        "method": "slow",
        "best_s": slow_samples.min(),
        "median_s": np.median(slow_samples),
        "std_s": slow_samples.std(ddof=1),
    },
    {
        "method": "fast",
        "best_s": fast_samples.min(),
        "median_s": np.median(fast_samples),
        "std_s": fast_samples.std(ddof=1),
    },
])

slow_median = benchmark_df.loc[
    benchmark_df["method"] == "slow",
    "median_s",
].iloc[0]

fast_median = benchmark_df.loc[
    benchmark_df["method"] == "fast",
    "median_s",
].iloc[0]

speedup_median = slow_median / fast_median

display(benchmark_df)

print(f"median speedup: {speedup_median:.2f}x")

### Task 3 - A compact acceleration decision report for a parameter sweep

The setup below defines a small family of one-dimensional fields. For each amplitude, we evolve the field for a few explicit steps and compute one diagnostic energy-like quantity.

In [ ]:
from pathlib import Path
import json
import timeit

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from numba import njit
    HAS_NUMBA = True
except ImportError:
    HAS_NUMBA = False


def make_initial_condition(n, amplitude):
    x = np.linspace(-10.0, 10.0, n)
    dx = x[1] - x[0]
    perturbation = amplitude * np.exp(-0.20*x**2) * np.sin(4.0*x)
    u = np.tanh(x) + perturbation
    return x, u, dx


def residual_reference_for_sweep(u, dx):
    r = np.zeros_like(u)
    inv_dx2 = 1.0 / dx**2

    for i in range(1, u.size - 1):
        u_xx = (u[i - 1] - 2.0*u[i] + u[i + 1]) * inv_dx2
        r[i] = -u_xx + u[i]*(u[i]**2 - 1.0)

    return r


def evolve_reference(u0, dx, dt, n_steps):
    u = u0.copy()

    for _ in range(n_steps):
        r = residual_reference_for_sweep(u, dx)
        u = u - dt * r

    return u


def energy_like(x, u):
    ux = np.gradient(u, x)
    density = 0.5*ux**2 + 0.25*(u**2 - 1.0)**2
    return np.trapezoid(density, x)


sweep_config = {
    "n": 6_000,
    "dt": 1.0e-6,
    "n_steps": 60,
    "amplitudes": [0.00, 0.04, 0.08, 0.12, 0.16, 0.20],
}

Your tasks are:

1. Implement `residual_out(u, dx, out)` and `evolve_reusing(u0, dx, dt, n_steps)` using one reusable residual array.
2. For one representative amplitude, validate `evolve_reusing` against `evolve_reference` by comparing the final field and the final value of `energy_like`.
3. Benchmark `evolve_reference` and `evolve_reusing` for the representative amplitude using repeated timings.
4. Run the full amplitude sweep with the faster validated workflow and collect a `DataFrame` with amplitude, final energy, maximum absolute field value, and runtime.
5. Save the summary table and `sweep_config` under `results_05/tables`.
6. Plot final energy versus amplitude.
7. If Numba is available, add it as an additional candidate only after the NumPy version has been validated. If it is not available, state that clearly.
8. Write a short decision report: what was measured, which bottleneck was addressed, what changed, whether the result stayed correct, and whether the added complexity was worth it.

In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Point 1
# Residual using one reusable output array
# ------------------------------------------------------------

def residual_out(u, dx, out):
    assert u.ndim == 1
    assert out.shape == u.shape
    assert u.size >= 3
    assert dx > 0.0
    assert np.isfinite(u).all()

    inv_dx2 = 1.0 / dx**2

    out[0] = 0.0
    out[-1] = 0.0

    out[1:-1] = (
        -(
            u[:-2]
            - 2.0 * u[1:-1]
            + u[2:]
        ) * inv_dx2
        + u[1:-1] * (u[1:-1]**2 - 1.0)
    )

    assert np.isfinite(out).all()

    return out


def evolve_reusing(u0, dx, dt, n_steps):
    assert u0.ndim == 1
    assert dx > 0.0
    assert dt > 0.0
    assert n_steps >= 0
    assert np.isfinite(u0).all()

    u = u0.copy()
    r = np.empty_like(u)

    for _ in range(n_steps):
        residual_out(u, dx, r)

        # Intentional in-place update of the evolving state.
        # The input u0 is not modified.
        u -= dt * r

    assert u.shape == u0.shape
    assert np.isfinite(u).all()

    return u

# ------------------------------------------------------------
# Point 2
# Validate evolve_reusing against evolve_reference
# ------------------------------------------------------------

representative_amplitude = 0.12

x_rep, u0_rep, dx_rep = make_initial_condition(
    sweep_config["n"],
    representative_amplitude,
)

u_ref = evolve_reference(
    u0_rep,
    dx_rep,
    sweep_config["dt"],
    sweep_config["n_steps"],
)

u_reuse = evolve_reusing(
    u0_rep,
    dx_rep,
    sweep_config["dt"],
    sweep_config["n_steps"],
)

energy_ref = energy_like(x_rep, u_ref)
energy_reuse = energy_like(x_rep, u_reuse)

validation = pd.DataFrame([
    {
        "quantity": "max_abs_field_difference",
        "value": np.max(np.abs(u_ref - u_reuse)),
    },
    {
        "quantity": "reference_energy",
        "value": energy_ref,
    },
    {
        "quantity": "reusing_energy",
        "value": energy_reuse,
    },
    {
        "quantity": "abs_energy_difference",
        "value": abs(energy_ref - energy_reuse),
    },
])

assert u_ref.shape == u_reuse.shape
assert np.isfinite(u_ref).all()
assert np.isfinite(u_reuse).all()
assert np.allclose(u_reuse, u_ref, rtol=1.0e-10, atol=1.0e-12)
assert np.allclose(energy_reuse, energy_ref, rtol=1.0e-10, atol=1.0e-12)

display(validation)

In [ ]:
# ------------------------------------------------------------
# Point 3
# Benchmark reference and reusable-output workflows
# ------------------------------------------------------------

def benchmark_call(call, *, number=1, repeat=5):
    samples = timeit.repeat(
        call,
        number=number,
        repeat=repeat,
    )

    return np.array(samples) / number


def benchmark_summary(method, samples):
    return {
        "method": method,
        "best_s": samples.min(),
        "median_s": np.median(samples),
        "std_s": samples.std(ddof=1),
        "repeat": samples.size,
    }


# Warm-up calls.
evolve_reference(
    u0_rep,
    dx_rep,
    sweep_config["dt"],
    sweep_config["n_steps"],
)

evolve_reusing(
    u0_rep,
    dx_rep,
    sweep_config["dt"],
    sweep_config["n_steps"],
)


reference_samples = benchmark_call(
    lambda: evolve_reference(
        u0_rep,
        dx_rep,
        sweep_config["dt"],
        sweep_config["n_steps"],
    ),
    number=1,
    repeat=5,
)

reusing_samples = benchmark_call(
    lambda: evolve_reusing(
        u0_rep,
        dx_rep,
        sweep_config["dt"],
        sweep_config["n_steps"],
    ),
    number=1,
    repeat=5,
)

benchmark_records = [
    benchmark_summary("reference", reference_samples),
    benchmark_summary("reusing_out_array", reusing_samples),
]

benchmark_df = pd.DataFrame(benchmark_records)

reference_median = benchmark_df.loc[
    benchmark_df["method"] == "reference",
    "median_s",
].iloc[0]

reusing_median = benchmark_df.loc[
    benchmark_df["method"] == "reusing_out_array",
    "median_s",
].iloc[0]

benchmark_df["speedup_vs_reference"] = reference_median / benchmark_df["median_s"]

display(benchmark_df)

In [ ]:
# ------------------------------------------------------------
# Point 4
# Run the full amplitude sweep with the faster validated workflow
# ------------------------------------------------------------

def run_one_amplitude(amplitude, config):
    x, u0, dx = make_initial_condition(
        config["n"],
        amplitude,
    )

    start = timeit.default_timer()

    u_final = evolve_reusing(
        u0,
        dx,
        config["dt"],
        config["n_steps"],
    )

    elapsed = timeit.default_timer() - start

    return {
        "amplitude": amplitude,
        "final_energy": energy_like(x, u_final),
        "max_abs_u": np.max(np.abs(u_final)),
        "runtime_s": elapsed,
    }


sweep_records = [
    run_one_amplitude(amplitude, sweep_config)
    for amplitude in sweep_config["amplitudes"]
]

sweep_df = pd.DataFrame(sweep_records)

assert sweep_df.shape[0] == len(sweep_config["amplitudes"])
assert np.isfinite(sweep_df[["final_energy", "max_abs_u", "runtime_s"]]).all(axis=None)

display(sweep_df)

In [ ]:
# ------------------------------------------------------------
# Point 5
# Save summary table and sweep configuration
# ------------------------------------------------------------

table_dir = Path("results_05") / "tables"
table_dir.mkdir(parents=True, exist_ok=True)

summary_path = table_dir / "sweep_summary.csv"
config_path = table_dir / "sweep_config.json"
benchmark_path = table_dir / "benchmark_summary.csv"

sweep_df.to_csv(summary_path, index=False)
benchmark_df.to_csv(benchmark_path, index=False)

config_path.write_text(
    json.dumps(sweep_config, indent=2),
    encoding="utf8",
)

saved_paths = {
    "summary": summary_path,
    "benchmark": benchmark_path,
    "config": config_path,
}

saved_paths

In [ ]:
# ------------------------------------------------------------
# Point 6
# Plot final energy versus amplitude
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.2, 4.0))

ax.plot(
    sweep_df["amplitude"],
    sweep_df["final_energy"],
    "o-",
)

ax.set_xlabel("amplitude")
ax.set_ylabel("final energy-like diagnostic")
ax.set_title("Final energy versus perturbation amplitude")
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Point 7
# Optional Numba candidate, only after NumPy version was validated
# ------------------------------------------------------------

if HAS_NUMBA:

    @njit
    def residual_numba_out(u, dx, out):
        inv_dx2 = 1.0 / dx**2

        out[0] = 0.0
        out[-1] = 0.0

        for i in range(1, u.size - 1):
            u_xx = (
                u[i - 1]
                - 2.0 * u[i]
                + u[i + 1]
            ) * inv_dx2

            out[i] = -u_xx + u[i] * (u[i]**2 - 1.0)


    @njit
    def evolve_numba(u0, dx, dt, n_steps):
        u = u0.copy()
        r = np.empty_like(u)

        for _ in range(n_steps):
            residual_numba_out(u, dx, r)

            for i in range(u.size):
                u[i] = u[i] - dt * r[i]

        return u


    # First call includes compilation. Do not use it as the benchmark.
    u_numba = evolve_numba(
        u0_rep,
        dx_rep,
        sweep_config["dt"],
        sweep_config["n_steps"],
    )

    energy_numba = energy_like(x_rep, u_numba)

    assert u_numba.shape == u_ref.shape
    assert np.isfinite(u_numba).all()
    assert np.allclose(u_numba, u_ref, rtol=1.0e-10, atol=1.0e-12)
    assert np.allclose(energy_numba, energy_ref, rtol=1.0e-10, atol=1.0e-12)

    numba_samples = benchmark_call(
        lambda: evolve_numba(
            u0_rep,
            dx_rep,
            sweep_config["dt"],
            sweep_config["n_steps"],
        ),
        number=1,
        repeat=5,
    )

    numba_record = benchmark_summary(
        "numba_reusing_out_array",
        numba_samples,
    )

    benchmark_df_with_numba = pd.concat(
        [
            benchmark_df,
            pd.DataFrame([numba_record]),
        ],
        ignore_index=True,
    )

    reference_median = benchmark_df_with_numba.loc[
        benchmark_df_with_numba["method"] == "reference",
        "median_s",
    ].iloc[0]

    benchmark_df_with_numba["speedup_vs_reference"] = (
        reference_median / benchmark_df_with_numba["median_s"]
    )

    benchmark_df_with_numba.to_csv(
        table_dir / "benchmark_summary_with_numba.csv",
        index=False,
    )

    numba_status = (
        "Numba is available. The JIT candidate was validated against the "
        "reference workflow and benchmarked after compilation."
    )

else:
    benchmark_df_with_numba = benchmark_df.copy()

    numba_status = (
        "Numba is not available in this environment, so no JIT candidate "
        "was benchmarked."
    )


display(benchmark_df_with_numba)

numba_status